# Mock Run — Iteration 2 Smoke Test

Exercises all Iteration 2 components with minimal compute:
- All 9 models (DLinear, PatchTST, BERTForecaster, LateFusion, GatedFusion, FiLMFusion, EnsembleFusion, **CrossAttentionFusion**, **ResidualCorrection**)
- Enriched text descriptions (MiniLM-L6, regime, autocorr, temporal context)
- Offline `text_emb` path (if `.npy` exists) and online fallback
- D-series ablation structure: template / random text sources
- Diagnostic logging (α, gate, β, attn weights)
- 1 epoch, `train_fraction=0.05`, `seq_len=96`, `pred_len=96`

**No Drive mount required.** Results are printed in-notebook only.

## 0. Setup

In [ ]:
import os, sys

# Detect project root
_cwd = os.getcwd()
if os.path.isfile(os.path.join(_cwd, 'run_mock.ipynb')):
    PROJECT_ROOT = _cwd
else:
    PROJECT_ROOT = _cwd

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)
print(f'Working directory: {os.getcwd()}')

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import torch

if torch.cuda.is_available():
    device_info = f'CUDA — {torch.cuda.get_device_name(0)}'
elif torch.backends.mps.is_available():
    device_info = 'Apple MPS'
else:
    device_info = 'CPU'

print(f'PyTorch : {torch.__version__}')
print(f'Device  : {device_info}')

## 1. Mock Settings

In [ ]:
MOCK_EPOCHS    = 1
MOCK_FRACTION  = 0.05   # 5% of train split
MOCK_SEQ_LEN   = 96     # shorter context window for speed
MOCK_PRED_LEN  = 96
MOCK_DATASET   = 'ETTh1'
MOCK_DATA_PATH = 'ETTh1.csv'
MOCK_ROOT_PATH = './dataset/ETT-small/'

# Path to pre-encoded embeddings (optional — skip if not present)
MOCK_EMB_PATH  = f'./dataset/embeddings/{MOCK_DATASET}_train_minilm.npy'
USE_OFFLINE_EMB = os.path.exists(MOCK_EMB_PATH)
print(f'Offline embeddings: {"FOUND" if USE_OFFLINE_EMB else "not found — using online encoding"}')

## 2. Run Helper

In [ ]:
from run_experiment import run, load_config, apply_overrides

mock_results = {}   # label → metrics dict or None

def mock_run(config_path: str, label: str, extra_overrides: list = None):
    """Run one mock experiment and store results."""
    overrides = [
        f'name=mock_{label}',
        f'model.seq_len={MOCK_SEQ_LEN}',
        f'model.pred_len={MOCK_PRED_LEN}',
        f'training.train_epochs={MOCK_EPOCHS}',
        f'training.train_fraction={MOCK_FRACTION}',
        f'training.patience=999',     # disable early stopping
        f'compute.num_workers=0',     # no multiprocessing in notebook
    ]
    if extra_overrides:
        overrides += extra_overrides

    print(f'\n{"="*55}')
    print(f'  {label}')
    print(f'{"="*55}')
    try:
        metrics = run(config_path, overrides=overrides)
        mock_results[label] = metrics
        print(f'  MAE={metrics["mae"]:.4f}  MSE={metrics["mse"]:.4f}')
        if metrics.get('diagnostics'):
            for k, v in metrics['diagnostics'].items():
                print(f'    diag/{k}: {v}')
    except Exception as e:
        print(f'  ERROR: {e}')
        mock_results[label] = None

print('Run helper ready.')

## 3. Baseline Models (no text)

In [ ]:
BASELINE_CONFIGS = [
    ('experiments/configs/01_dlinear_etth1.yaml',  'dlinear'),
    ('experiments/configs/02_patchtst_etth1.yaml', 'patchtst'),
]

for cfg_path, label in BASELINE_CONFIGS:
    mock_run(cfg_path, label)

## 4. Text-Augmented Models — Online Encoding (template)

Exercises enriched MiniLM-L6 descriptions: regime label, autocorr lag-1/24, temporal context.

In [ ]:
TEXT_CONFIGS = [
    ('experiments/configs/03_bert_forecaster_etth1.yaml', 'bert_forecaster'),
    ('experiments/configs/04_late_fusion_etth1.yaml',     'late_fusion'),
    ('experiments/configs/05_gated_fusion_etth1.yaml',    'gated_fusion'),
    ('experiments/configs/06_film_fusion_etth1.yaml',     'film_fusion'),
    ('experiments/configs/07_ensemble_fusion_etth1.yaml', 'ensemble_fusion'),
]

for cfg_path, label in TEXT_CONFIGS:
    mock_run(cfg_path, label, extra_overrides=['model.text_source=template'])

## 5. New Models — F8 CrossAttentionFusion & F10 ResidualCorrection

In [ ]:
import yaml
from pathlib import Path

# Build minimal configs inline (no pre-existing YAML needed)
_NEW_MODEL_BASE = {
    'data': {
        'dataset': MOCK_DATASET,
        'root_path': MOCK_ROOT_PATH,
        'data_path': MOCK_DATA_PATH,
        'freq': 'h',
        'features': 'M',
        'target': 'OT',
    },
    'training': {
        'train_epochs': MOCK_EPOCHS,
        'batch_size': 16,
        'learning_rate': 0.0001,
        'patience': 999,
        'seed': 2024,
        'train_fraction': MOCK_FRACTION,
    },
    'compute': {'gpu': 0, 'num_workers': 0, 'use_amp': False},
}

_NEW_MODEL_CFG = {
    'task_name': 'long_term_forecast',
    'enc_in': 7,
    'seq_len': MOCK_SEQ_LEN,
    'label_len': 24,
    'pred_len': MOCK_PRED_LEN,
    'd_model': 64,
    'n_heads': 4,
    'e_layers': 2,
    'd_layers': 1,
    'd_ff': 256,
    'factor': 1,
    'dropout': 0.1,
    'activation': 'gelu',
    'text_model': 'sentence-transformers/all-MiniLM-L6-v2',
    'text_hidden': 384,
    'text_source': 'template',
}

tmp_dir = Path('experiments/configs/mock_tmp')
tmp_dir.mkdir(parents=True, exist_ok=True)

for model_name, label, diag_flags in [
    ('CrossAttentionFusion', 'cross_attn', {'enabled': True, 'log_attn': True}),
    ('ResidualCorrection',   'residual_correction', {'enabled': True, 'log_beta': True}),
]:
    cfg = {
        **_NEW_MODEL_BASE,
        'name': f'mock_{label}',
        'model': {**_NEW_MODEL_CFG, 'name': model_name},
        'diagnostics': diag_flags,
    }
    cfg_path = tmp_dir / f'mock_{label}.yaml'
    with open(cfg_path, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)
    mock_run(str(cfg_path), label)

## 6. D-Series Ablation — Random Text Source

Verifies that `text_source=random` replaces MiniLM-L6 with `torch.randn` (D3/A1 ablation).

In [ ]:
for model_name, label, diag_flags in [
    ('GatedFusion',          'gated_random',  {'enabled': True, 'log_gates': True}),
    ('CrossAttentionFusion', 'cross_random',  {'enabled': True, 'log_attn': True}),
]:
    cfg = {
        **_NEW_MODEL_BASE,
        'name': f'mock_{label}',
        'model': {**_NEW_MODEL_CFG, 'name': model_name, 'text_source': 'random'},
        'diagnostics': diag_flags,
    }
    cfg_path = tmp_dir / f'mock_{label}.yaml'
    with open(cfg_path, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)
    mock_run(str(cfg_path), label)

## 7. Offline Embedding Mode (if `.npy` available)

Verifies the 5-tuple DataLoader path when `data.text_emb_path` is set.
Skip this cell if the `.npy` file has not been generated yet.

In [ ]:
if not USE_OFFLINE_EMB:
    print(f'Skipping offline embedding test — {MOCK_EMB_PATH} not found.')
    print('Run utils/text/encode_descriptions.py first (Section 3 of iteration2_run_experiment.ipynb).')
else:
    for model_name, label in [
        ('GatedFusion', 'gated_offline'),
        ('ResidualCorrection', 'residual_offline'),
    ]:
        cfg = {
            **_NEW_MODEL_BASE,
            'name': f'mock_{label}',
            'model': {**_NEW_MODEL_CFG, 'name': model_name, 'text_source': 'template'},
            'data': {**_NEW_MODEL_BASE['data'], 'text_emb_path': MOCK_EMB_PATH},
        }
        cfg_path = tmp_dir / f'mock_{label}.yaml'
        with open(cfg_path, 'w') as f:
            yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)
        mock_run(str(cfg_path), label)

## 8. Diagnostic Logging — EnsembleFusion α

Verifies that `diagnostics.log_alpha` causes the registry entry to include `diagnostics.alpha_mean/std`.

In [ ]:
cfg = {
    **_NEW_MODEL_BASE,
    'name': 'mock_ensemble_diag',
    'model': {**_NEW_MODEL_CFG, 'name': 'EnsembleFusion', 'text_source': 'template'},
    'diagnostics': {'enabled': True, 'log_alpha': True},
}
cfg_path = tmp_dir / 'mock_ensemble_diag.yaml'
with open(cfg_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

mock_run(str(cfg_path), 'ensemble_diag')

# Verify diagnostic field
m = mock_results.get('ensemble_diag')
if m and m.get('diagnostics'):
    print('✓ diagnostics field present:', list(m['diagnostics'].keys()))
else:
    print('✗ diagnostics field missing — check exp_forecasting.py test() method')

## 9. Results Summary

In [ ]:
import pandas as pd

rows = []
for label, metrics in mock_results.items():
    if metrics is None:
        rows.append({'model': label, 'MAE': None, 'MSE': None, 'status': 'ERROR'})
    else:
        rows.append({
            'model':  label,
            'MAE':    round(metrics.get('mae', float('nan')), 4),
            'MSE':    round(metrics.get('mse', float('nan')), 4),
            'diags':  ', '.join(metrics.get('diagnostics', {}).keys()) or '—',
            'status': 'OK',
        })

df = pd.DataFrame(rows)
print(f'\n=== Mock Run | {MOCK_DATASET} | pred_len={MOCK_PRED_LEN} | {MOCK_EPOCHS} epoch | fraction={MOCK_FRACTION} ===')
print(df.to_string(index=False))

n_ok  = (df['status'] == 'OK').sum()
n_err = (df['status'] == 'ERROR').sum()
print(f'\n{n_ok} passed, {n_err} failed')

## 10. Cleanup Temp Configs (optional)

In [ ]:
import shutil
shutil.rmtree(tmp_dir, ignore_errors=True)
print('Temp configs removed.')